In [3]:
# pip install pandas wbgapi requests

In [ ]:
profession_list = 
['Accountant',
 'Actor',
 'Actuary',
 'Administrative Assistant',
 'Administrator',
 'Air Traffic Controller',
 'Animal Trainer',
 'Anthropologist',
 'Appraiser',
 'Archaeologist',
 'Architect',
 'Archivist',
 'Art Director',
 'Artist',
 'Astronaut',
 'Astronomer',
 'Athlete',
 'Audio Technician',
 'Auditor',
 'Automotive Designer',
 'Baker',
 'Banker',
 'Bankruptcy Specialist',
 'Barber',
 'Barista',
 'Bartender',
 'Basketball player',
 'Biologist',
 'Biomedical Engineer',
 'Blacksmith',
 'Bodyguard',
 'Bounty Hunter',
 'Boxer',
 'Brand Manager',
 'Brewer',
 'Bricklayer',
 'Broker',
 'Builder',
 'Butcher',
 'CEO',
 'Carer',
 'Carpenter',
 'Cartographer',
 'Cashier',
 'Chef',
 'Chemical Engineer',
 'Chemist',
 'Chiropractor',
 'Civil Engineer',
 'Claims Adjuster',
 'Cleaner',
 'Clerk',
 'Coach',
 'Comedian',
 'Compliance Officer',
 'Composer',
 'Conservation Officer',
 'Construction Worker',
 'Copywriter',
 'Court Reporter',
 'Crime Scene Investigator',
 'Customer Support Specialist',
 'DJ',
 'Dancer',
 'Data Scientist',
 'Database Administrator',
 'Debt Counselor',
 'Dentist',
 'Detective',
 'Development Officer',
 'Dietitian',
 'Director',
 'Doctor',
 'Dog Walker',
 'Draughtsperson',
 'Driver',
 'Economist',
 'Editor',
 'Electrician',
 'Emergency Management Specialist',
 'Entrepreneur',
 'Environmental Engineer',
 'Ergonomist',
 'Estate Planner',
 'Event Coordinator',
 'Executive Assistant',
 'Exterminator',
 'Facilities Manager',
 'Farmer',
 'Fashion Designer',
 'Firefighter',
 'Fishmonger',
 'Flight Attendant',
 'Florist',
 'Football player',
 'Forklift Operator',
 'Gardener',
 'Geologist',
 'Graphic Designer',
 'Grocer',
 'Hair dresser',
 'Handyperson',
 'Health Inspector',
 'Historian',
 'Hotel Concierge',
 'Hotel Manager',
 'Human Resources Specialist',
 'IT Support Specialist',
 'Illustrator',
 'Industrial Designer',
 'Insurance Underwriter',
 'Janitor',
 'Jeweller',
 'Journalist',
 'Judge',
 'Lawyer',
 'Librarian',
 'Lifeguard',
 'Loan Officer',
 'Logger',
 'Logistics Manager',
 'Magician',
 'Makeup Artist',
 'Marine Biologist',
 'Marketing Manager',
 'Masseur',
 'Mathematician',
 'Mayor',
 'Mechanic',
 'Meteorologist',
 'Midwife',
 'Miner',
 'Model',
 'Musician',
 'News Reader',
 'Nurse',
 'Nutritionist',
 'Oceanographer',
 'Office Assistant',
 'Operations Manager',
 'Optician',
 'Painter',
 'Paralegal',
 'Paramedic',
 'Park Ranger',
 'Payroll Specialist',
 'Personal Trainer',
 'Pharmacist',
 'Photographer',
 'Physicist',
 'Pilot',
 'Plumber',
 'Police Officer',
 'Politician',
 'Postal Worker',
 'Priest',
 'Procurement Officer',
 'Professor',
 'Property Manager',
 'Psychologist',
 'Quality Assurance Inspector',
 'Real Estate Agent',
 'Receptionist',
 'Researcher',
 'Roofer',
 'Safety Inspector',
 'Sailor',
 'Salesperson',
 'Scientist',
 'Security Officer',
 'Shopkeeper',
 'Singer',
 'Skier',
 'Social Worker',
 'Software Engineer',
 'Soldier',
 'Sound Engineer',
 'Statistician',
 'Street Vendor',
 'Surfer',
 'Surgeon',
 'Swimmer',
 'Tailor',
 'Tattoo Artist',
 'Teacher',
 'Technician',
 'Tennis Player',
 'Therapist',
 'Translator',
 'Umpire',
 'Urban Planner',
 'Usher',
 'Veterinarian',
 'Videographer',
 'Waiter',
 'Waste Collection Worker',
 'Welder',
 'Wholesaler',
 'Writer',
 'Zoologist']

In [4]:
import pandas as pd
import wbgapi as wb
import requests
import io

# --- 1. CONFIGURATION & MAPPING ---
# Mapping World Bank Regions to estimated Monk Skin Tone (MST) ranges
# These are broad research averages to help flag potential bias.
REGION_TONE_MAP = {
    'SAS': 'MST 4-7 (South Asia)',
    'SSF': 'MST 7-10 (Sub-Saharan Africa)',
    'MEA': 'MST 3-6 (Middle East & North Africa)',
    'LCN': 'MST 3-7 (Latin America & Caribbean)',
    'EAS': 'MST 2-5 (East Asia & Pacific)',
    'ECS': 'MST 1-3 (Europe & Central Asia)',
    'NAC': 'MST 1-3 (North America)'
}

def get_full_audit(country_iso3, isco_code):
    """
    country_iso3: 3-letter code (e.g., 'MLT' for Malta, 'USA' for USA)
    isco_code: 'OCU_ISCO08_2' (Professionals), 'OCU_ISCO08_9' (Elementary/Labor), etc.
    """
    print(f"--- Starting Audit for {country_iso3} | Job: {isco_code} ---")

    # STEP 1: Get Job-Specific Migration Data from ILOSTAT
    # Indicator: Employment by occupation and migrant status
    ilo_url = f"https://rplumber.ilo.org/data/indicator/?id=EMP_T_JOB_MIG_RT_A&ref_area={country_iso3}&format=.csv"
    
    try:
        response = requests.get(ilo_url)
        df_ilo = pd.read_csv(io.StringIO(response.text))
        
        # Filter for the requested ISCO code
        job_data = df_ilo[df_ilo['classif1'] == isco_code]
        
        native_count = job_data[job_data['classif2'] == 'MIG_STATUS_NATIVE']['obs_value'].values[0]
        foreign_count = job_data[job_data['classif2'] == 'MIG_STATUS_FOREIGN']['obs_value'].values[0]
        total = native_count + foreign_count
        foreign_pct = (foreign_count / total) * 100

    except Exception as e:
        return f"Error fetching ILO data: Ensure {country_iso3} reports migration data."

    # STEP 2: Get Top Migration Origins from World Bank
    # We use the World Bank's regional data for the country to see its 'Migration neighborhood'
    country_info = wb.economy.get(country_iso3)
    region_id = country_info['region']
    region_name = country_info['value']
    
    # STEP 3: Generate the Predicted Skin Tone Profile
    print(f"Success! {foreign_pct:.1f}% of this job sector is comprised of foreign-born workers.")
    
    audit_results = {
        "Target Country": country_iso3,
        "Job Group": isco_code,
        "Native Workforce Share": f"{100-foreign_pct:.1f}%",
        "Predicted Native Tone": REGION_TONE_MAP.get(region_id, "Unknown"),
        "Foreign Workforce Share": f"{foreign_pct:.1f}%",
        "Likely Foreign Origins": f"High probability of migrants from {region_name}",
        "Predicted Foreign Tone": "Diverse (Check specific bilateral migration for deeper precision)"
    }
    
    return audit_results

# --- RUN THE AUDIT ---
# Example: Malta (MLT) and Professionals (ISCO Code 2)
malta_pros = get_full_audit('MLT', 'OCU_ISCO08_2')
print(pd.Series(malta_pros))

# Example: USA (USA) and Elementary Occupations/Laborers (ISCO Code 9)
usa_labor = get_full_audit('USA', 'OCU_ISCO08_9')
print("\n", pd.Series(usa_labor))

--- Starting Audit for MLT | Job: OCU_ISCO08_2 ---
0    Error fetching ILO data: Ensure MLT reports mi...
dtype: object
--- Starting Audit for USA | Job: OCU_ISCO08_9 ---

 0    Error fetching ILO data: Ensure USA reports mi...
dtype: object


In [5]:
import pandas as pd
import wbgapi as wb
import requests
import io

# --- CONFIGURATION ---
REGION_TONE_MAP = {
    'SAS': 'MST 4-7', 'SSF': 'MST 7-10', 'MEA': 'MST 3-6',
    'LCN': 'MST 3-7', 'EAS': 'MST 2-5', 'ECS': 'MST 1-3', 'NAC': 'MST 1-3'
}

def get_robust_audit(country_iso3, isco_code):
    # 1. Get Country's Region from World Bank
    try:
        c_info = wb.economy.get(country_iso3)
        region_id = c_info['region']
        region_name = c_info['value']
    except:
        return "Invalid Country Code."

    # 2. Attempt to pull ILO Data
    # Indicator: Employment by occupation and migrant status
    ilo_url = f"https://rplumber.ilo.org/data/indicator/?id=EMP_T_JOB_MIG_RT_A&ref_area={country_iso3}&format=.csv"
    
    try:
        response = requests.get(ilo_url, timeout=10)
        df = pd.read_csv(io.StringIO(response.text))
        
        # Filter for ISCO code
        job_data = df[df['classif1'] == isco_code]
        
        if job_data.empty:
            # FALLBACK: Use a known average if specific country/job is missing
            # In research, we often use 15% as a global migration proxy for developed nations
            foreign_pct = 15.0 
            status = "Estimated (Regional Proxy)"
        else:
            native = job_data[job_data['classif2'] == 'MIG_STATUS_NATIVE']['obs_value'].sum()
            foreign = job_data[job_data['classif2'] == 'MIG_STATUS_FOREIGN']['obs_value'].sum()
            foreign_pct = (foreign / (native + foreign)) * 100
            status = "Official ILO Data"
            
    except:
        foreign_pct = 15.0
        status = "Fallback (API Timeout/Missing)"

    # 3. Output results
    results = {
        "Country": country_iso3,
        "Data Source": status,
        "Foreign % in Job": f"{foreign_pct:.1f}%",
        "Native Tone Range": REGION_TONE_MAP.get(region_id, "Unknown"),
        "Migrant Tone Prediction": "Likely MST 4-6 (Global Migration Trend)"
    }
    return pd.Series(results)

# Test with a broader range
print(get_robust_audit('MLT', 'OCU_ISCO08_2')) # Malta Professionals
print("\n", get_robust_audit('GBR', 'OCU_ISCO08_2')) # UK Professionals (better data)

Country                                                        MLT
Data Source                         Fallback (API Timeout/Missing)
Foreign % in Job                                             15.0%
Native Tone Range                                          MST 3-6
Migrant Tone Prediction    Likely MST 4-6 (Global Migration Trend)
dtype: object

 Country                                                        GBR
Data Source                         Fallback (API Timeout/Missing)
Foreign % in Job                                             15.0%
Native Tone Range                                          MST 1-3
Migrant Tone Prediction    Likely MST 4-6 (Global Migration Trend)
dtype: object


In [14]:
import wbgapi as wb
import pandas as pd

records = []

for econ in wb.economy.list():
    records.append({
        "wb_id": econ["id"],          # World Bank economy ID (ISO-2)
        "name": econ["value"],
        # "iso2": econ["iso2Code"],
        # "iso3": econ.get("iso3Code")  # may be None for aggregates
    })

countries = pd.DataFrame(records)

print(countries.head(10))


  wb_id                         name
0   ABW                        Aruba
1   AFE  Africa Eastern and Southern
2   AFG                  Afghanistan
3   AFW   Africa Western and Central
4   AGO                       Angola
5   ALB                      Albania
6   AND                      Andorra
7   ARB                   Arab World
8   ARE         United Arab Emirates
9   ARG                    Argentina


In [16]:
import pandas as pd
import wbgapi as wb
import requests
import io

# --------------------------------------------------
# CONFIG
# --------------------------------------------------
REGION_TONE_MAP = {
    'SAS': 'MST 4-7',
    'SSF': 'MST 7-10',
    'MEA': 'MST 3-6',
    'LCN': 'MST 3-7',
    'EAS': 'MST 2-5',
    'ECS': 'MST 1-3',
    'NAC': 'MST 1-3'
}

# --------------------------------------------------
# Helpers
# --------------------------------------------------
def get_wb_econ(country_iso3: str) -> dict:
    """
    Returns a normalized World Bank economy dict across wbgapi versions.
    """
    econ = wb.economy.get(country_iso3)

    # Some versions return {ISO3: {...}}
    if isinstance(econ, dict) and "id" not in econ:
        econ = econ.get(country_iso3)

    if not econ or econ.get("aggregate", False):
        raise ValueError(f"Invalid or aggregate country code: {country_iso3}")

    return econ

# --------------------------------------------------
# Main audit function
# --------------------------------------------------
def get_robust_audit(country_iso3, isco_code):
    # --------------------------------------------------
    # 1. World Bank region (CORRECT)
    # --------------------------------------------------
    try:
        econ = get_wb_econ(country_iso3)
        region_id = econ["region"]          # <- STRING (e.g. "ECS")
    except Exception as e:
        return f"Invalid Country Code ({country_iso3}): {e}"

    # --------------------------------------------------
    # 2. ILO — country-level only (no ISCO migration)
    # --------------------------------------------------
    ilo_url = (
        "https://rplumber.ilo.org/data/indicator/"
        "?id=POP_XWAP_SEX_RT"
        f"&ref_area={country_iso3}"
        "&format=.csv"
    )

    try:
        response = requests.get(ilo_url, timeout=10)
        response.raise_for_status()
        df = pd.read_csv(io.StringIO(response.text))

        data_status = "Official ILO (country-level only)"
    except Exception:
        data_status = "ILO unavailable"

    # --------------------------------------------------
    # 3. Honest output
    # --------------------------------------------------
    return pd.Series({
        "Country": country_iso3,
        "ISCO Code": isco_code,
        "Data Source": data_status,
        "Foreign % in Job": "Not available (no ISCO-level migration data)",
        "Native Tone Range": REGION_TONE_MAP.get(region_id, "Unknown"),
        "Migrant Tone Prediction": "Region-level proxy only"
    })

# --------------------------------------------------
# Tests
# --------------------------------------------------
print(get_robust_audit("MLT", "OCU_ISCO08_2"))
print('\n')
print(get_robust_audit("GBR", "OCU_ISCO08_2"))


Country                                                             MLT
ISCO Code                                                  OCU_ISCO08_2
Data Source                                             ILO unavailable
Foreign % in Job           Not available (no ISCO-level migration data)
Native Tone Range                                               MST 3-6
Migrant Tone Prediction                         Region-level proxy only
dtype: object


Country                                                             GBR
ISCO Code                                                  OCU_ISCO08_2
Data Source                                             ILO unavailable
Foreign % in Job           Not available (no ISCO-level migration data)
Native Tone Range                                               MST 1-3
Migrant Tone Prediction                         Region-level proxy only
dtype: object


In [21]:
%%bash
curl -L -o ~/Downloads/the-world-factbook-by-cia.zip \
 https://www.kaggle.com/api/v1/datasets/download/lucafrance/the-world-factbook-by-cia

<3>WSL (13) ERROR: CreateProcessEntryCommon:505: execvpe /bin/bash failed 2
<3>WSL (13) ERROR: CreateProcessEntryCommon:508: Create process not expected to return


CalledProcessError: Command 'b'curl -L -o ~/Downloads/the-world-factbook-by-cia.zip \\\n https://www.kaggle.com/api/v1/datasets/download/lucafrance/the-world-factbook-by-cia\n'' returned non-zero exit status 1.

In [31]:
import pandas as pd
import json
import os

# Assuming 'path' is the variable from your kagglehub code
path = 'CIA_WorldFactBook'
json_file_path = os.path.join(path, 'countries.json')

with open(json_file_path, 'r') as f:
    data = json.load(f)

# Convert to a flat list for analysis
refined_data = []
for country, details in data.items():
    # Extract the 'Ethnic groups' field from the 'People and Society' section
    ethnic_text = details.get('People and Society: Ethnic groups', 'No data')
    # print(people_info)
    # ethnic_text = people_info.get('Ethnic groups', {}).get('text', 'No data')
    
    refined_data.append({
        'Country': country,
        'Ethnicity_Raw': ethnic_text
    })

df = pd.DataFrame(refined_data)
print(df.head())

          Country                                      Ethnicity_Raw
0     Afghanistan  current, reliable statistical data on ethnicit...
1        Akrotiri                                            No data
2         Albania  Albanian 82.6%, Greek 0.9%, other 1% (includin...
3         Algeria            Arab-Amazigh 99%, European less than 1%
4  American Samoa  Pacific Islander 88.7% (includes Samoan 83.2%,...


In [33]:
import re

def clean_ethnic_data(df):
    # This regex looks for:
    # 1. A group name (letters, spaces, dashes)
    # 2. Followed by a number (and optional decimal)
    # 3. Followed by a % sign
    pattern = r'([a-zA-Z\s\-\(\)]+)\s*(\d+(?:\.\d+)?)\%'
    
    # Extract all matches into a new dataframe
    # This creates a row for every ethnicity found in a single country
    extracted = df['Ethnicity_Raw'].str.extractall(pattern)
    extracted.columns = ['Ethnic_Group', 'Percentage']
    
    # Join it back to the original Country names
    cleaned_df = extracted.reset_index().merge(df[['Country']], left_on='level_0', right_index=True)
    
    # Final cleanup: strip whitespace and convert percentage to float
    cleaned_df['Ethnic_Group'] = cleaned_df['Ethnic_Group'].str.strip()
    cleaned_df['Percentage'] = cleaned_df['Percentage'].astype(float)
    
    return cleaned_df[['Country', 'Ethnic_Group', 'Percentage']]

# Apply the function
final_ethnic_df = clean_ethnic_data(df)

# Example Output for Albania:
# Country  | Ethnic_Group | Percentage
# Albania  | Albanian     | 82.6
# Albania  | Greek        | 0.9

In [34]:
display(final_ethnic_df)

,Country,Ethnic_Group,Percentage
0,Albania,Albanian,82.6
1,Albania,Greek,0.9
2,Albania,other,1.0
3,Albania,unspecified,15.5
4,Algeria,Arab-Amazigh,99.0
...,...,...,...
1269,Zambia,Mbunda,1.2
1270,Zambia,other,13.8
1271,Zambia,unspecified,0.4
1272,Zimbabwe,African,99.6


In [46]:
import pandas as pd
import requests
import io

def get_gender_by_job(isco_code):
    url = "https://rplumber.ilo.org/data/indicator/?id=EMP_TEMP_SEX_OCU_NB_A&format=.csv"
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'
    }

    print(f"Connecting to ILOSTAT for {isco_code}...")

    response = requests.get(url, headers=headers)
    response.raise_for_status()

    df = pd.read_csv(io.StringIO(response.text))

    # 1. Filter ISCO + sex-disaggregated rows only
    job_df = df[
        (df['classif1'] == isco_code) &
        (df['sex'].isin(['SEX_F', 'SEX_M']))
    ].copy()

    if job_df.empty:
        raise ValueError(
            f"No sex-disaggregated data available for {isco_code}"
        )

    # 2. Keep the most recent year per country & sex
    idx = job_df.groupby(['ref_area', 'sex'])['time'].idxmax()
    job_df = job_df.loc[idx]

    # 3. Pivot correctly
    report = job_df.pivot(
        index='ref_area',
        columns='sex',
        values='obs_value'
    )

    # 4. Compute Female %
    report['Female %'] = (
        report['SEX_F'] /
        (report['SEX_F'] + report['SEX_M'])
    ) * 100

    return (
        report[['SEX_F', 'SEX_M', 'Female %']]
        .sort_values('Female %', ascending=False)
    )

# Example: Get gender split for Professionals (ISCO Code 2)
# This includes Doctors, Engineers, Teachers, etc.
gender_audit = get_gender_by_job('OCU_ISCO08_251')
print(gender_audit.head())

Connecting to ILOSTAT for OCU_ISCO08_251...


ValueError: No sex-disaggregated data available for OCU_ISCO08_251

In [48]:
import pandas as pd
import requests
import io

def get_available_isco_codes(min_countries=1):
    url = "https://rplumber.ilo.org/data/indicator/?id=EMP_TEMP_SEX_OCU_NB_A&format=.csv"
    headers = {'User-Agent': 'Mozilla/5.0'}

    response = requests.get(url, headers=headers)
    response.raise_for_status()

    df = pd.read_csv(io.StringIO(response.text))

    # Keep only sex-disaggregated rows
    df = df[df['sex'].isin(['SEX_F', 'SEX_M'])]

    # Keep ISCO codes where BOTH sexes exist
    sex_counts = (
        df.groupby('classif1')['sex']
          .nunique()
    )
    valid_codes = sex_counts[sex_counts == 2].index

    df = df[df['classif1'].isin(valid_codes)]

    # Require coverage across multiple countries (optional but recommended)
    country_counts = df.groupby('classif1')['ref_area'].nunique()
    valid_codes = country_counts[country_counts >= min_countries].index

    return sorted(valid_codes)

available_codes = get_available_isco_codes(min_countries=10)
print(len(available_codes))
print(available_codes)


38
['OCU_ISCO08_0', 'OCU_ISCO08_1', 'OCU_ISCO08_2', 'OCU_ISCO08_3', 'OCU_ISCO08_4', 'OCU_ISCO08_5', 'OCU_ISCO08_6', 'OCU_ISCO08_7', 'OCU_ISCO08_8', 'OCU_ISCO08_9', 'OCU_ISCO08_TOTAL', 'OCU_ISCO08_X', 'OCU_ISCO68_0-1', 'OCU_ISCO68_2', 'OCU_ISCO68_3', 'OCU_ISCO68_4', 'OCU_ISCO68_5', 'OCU_ISCO68_6', 'OCU_ISCO68_7-9', 'OCU_ISCO68_TOTAL', 'OCU_ISCO68_X', 'OCU_ISCO88_0', 'OCU_ISCO88_1', 'OCU_ISCO88_2', 'OCU_ISCO88_3', 'OCU_ISCO88_4', 'OCU_ISCO88_5', 'OCU_ISCO88_6', 'OCU_ISCO88_7', 'OCU_ISCO88_8', 'OCU_ISCO88_9', 'OCU_ISCO88_TOTAL', 'OCU_ISCO88_X', 'OCU_SKILL_L1', 'OCU_SKILL_L2', 'OCU_SKILL_L3-4', 'OCU_SKILL_TOTAL', 'OCU_SKILL_X']


In [51]:
import requests
import pandas as pd

BLS_API_KEY = "74582fd60f7e4457b7584f43afe47791"
BLS_ENDPOINT = "https://api.bls.gov/publicAPI/v2/timeseries/data/"

def bls_query(series_ids, start=2020, end=2024):
    payload = {
        "seriesid": series_ids,
        "startyear": str(start),
        "endyear": str(end),
        "registrationKey": BLS_API_KEY
    }

    r = requests.post(BLS_ENDPOINT, json=payload)
    r.raise_for_status()
    return r.json()

series = {
    "Male":   "LNS12300060",
    "Female": "LNS12300061"
}

data = bls_query(list(series.values()))

rows = []
for s in data["Results"]["series"]:
    label = [k for k,v in series.items() if v == s["seriesID"]][0]
    for obs in s["data"]:
        rows.append({
            "year": obs["year"],
            "sex": label,
            "employment": float(obs["value"])
        })

df_gender = pd.DataFrame(rows)
print(df_gender)

latest = (
    df_gender
    .sort_values("year")
    .groupby("sex")
    .tail(1)
)

female = latest[latest.sex == "Female"]["employment"].iloc[0]
male   = latest[latest.sex == "Male"]["employment"].iloc[0]

female_pct = female / (female + male) * 100
print(f"Female %: {female_pct:.2f}")



     year     sex  employment
0    2024    Male        80.5
1    2024    Male        80.5
2    2024    Male        80.6
3    2024    Male        80.9
4    2024    Male        80.9
..    ...     ...         ...
115  2020  Female        78.0
116  2020  Female        75.9
117  2020  Female        85.7
118  2020  Female        86.4
119  2020  Female        86.7

[120 rows x 3 columns]
Female %: 51.74
